In [1]:
import os, subprocess, shutil
from os import path
from pathlib import Path
import numpy as np
import torch
import clip
from PIL import Image
from tqdm.notebook import tqdm
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform
from scipy.signal import savgol_filter
import matplotlib.pyplot as plt
import json

import itertools
from scipy.cluster.hierarchy import linkage, fcluster

print(f"✅ PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

✅ PyTorch: 2.10.0+cu128, CUDA: True
GPU: NVIDIA GeForce RTX 5090


In [2]:
# # Event Segmentation Pipeline for Toy Car Disassembly and Assembly
# 
# ## 🎯 Task Objectives
# - **Atomic Events**: Removing/installing a single screw, removing/installing a single component (3–8 seconds)
# - **Mid-level events**: Removing/installing the left wheel, right wheel, body, etc. (15–30 seconds)
# - **Top-level events**: Disassembling the toy car vs. assembling the toy car (1–2.5 minutes)

# ## 🔧 Environment Setup

# ## 📹 Configuration Parameters

# ========== Basic Configuration ==========
VIDEO_PATH = "NExTVideo"
OUTPUT_DIR = "pipeline_output"
PREPROCESSED_DIR = "preprocessed_frames"
FPS = 10
BATCH_SIZE = 256
CLEAN_FRAMES = False
REUSE_EXISTING_FRAMES = True

# ========== Segmentation Strategy ==========
# Complete Enhanced Segmentation Process:
# 1. Asymmetric sliding window (adaptive parameter search)
#    - Window Size: 10/12/15 frames
#    - Threshold: 0.72/0.74/0.76/0.80
#    - Similarity Method: Cosine
# 2. Traditional Differential Peak Detection (85th percentile, Savitzky-Golay filtering)
# 3. Merge two candidate tracks (deduplicate with a minimum interval of 0.3s)
# 4. Refine boundaries (select the point with the maximum gradient within ±0.3s)
# 5. Quality fallback (select the best single method when score < 0.5)

# ========== Hierarchical Configuration ==========
HIERARCHY_CONFIG = {
    'sigma': 30.0,  # Time-scale bandwidth (seconds) – Reduce to enhance locality
    'cut_levels': {
        'atomic': None,           # Atomic level (no segmentation)
        'component': 70,          # Component level (e.g., single wheel)
        'side': 85,               # Side level (e.g., left/right)
        'activity': 95            # Activity level (disassembly/assembly)
    }
}

# Get all filenames in the video path directory


# # ========== Path Settings ==========
device = "cuda" if torch.cuda.is_available() else "cpu"

# print(f"\n🎬 Video: {video_name}")
# print(f"📁 Output: {output_dir}")
# print(f"🔧 Device: {device}, FPS: {FPS}")
# print(f"🎯 Strategy: Complete Enhanced Version (Asymmetric Window + Differential Peak + Fusion + Refinement)")

In [3]:
def compute_window_similarity(window_a, window_b, method='cosine'):
    """Calculate the similarity between two windows"""
    if method == 'cosine':
        mean_a = np.mean(window_a, axis=0)
        mean_b = np.mean(window_b, axis=0)
        na, nb = np.linalg.norm(mean_a), np.linalg.norm(mean_b)
        if na == 0 or nb == 0:
            return 0.0
        return float(np.dot(mean_a, mean_b) / (na * nb))
    elif method == 'l2':
        mean_a = np.mean(window_a, axis=0)
        mean_b = np.mean(window_b, axis=0)
        return float(np.exp(-np.linalg.norm(mean_a - mean_b)))
    elif method == 'correlation':
        mean_a = np.mean(window_a, axis=0)
        mean_b = np.mean(window_b, axis=0)
        if np.std(mean_a) == 0 or np.std(mean_b) == 0:
            return 1.0 if np.allclose(mean_a, mean_b) else 0.0
        r = np.corrcoef(mean_a, mean_b)[0, 1]
        return float(r) if not np.isnan(r) else 0.0
    return 0.0


def sw_asymmetric(feats, window_size=10, step_reset=1, threshold=0.8,
                  similarity_method='cosine', min_segment_frames=10):
    """Asymmetric sliding window segmentation
    
    Right window fixed at window_size; the left window starts at window_size and increments by 1 each time,
    until it reaches the upper limit (3*window) or a split is triggered, at which point it resets; reset step size = 1.
    """
    T = feats.shape[0]
    right_size = int(window_size)
    left_init = int(window_size)
    grow = 1
    max_left = right_size * 3
    
    boundaries, similarities, positions = [], [], []
    pos, left_len = 0, left_init
    
    while pos + left_len + right_size <= T:
        left_start = pos
        left_end = pos + left_len
        right_end = left_end + right_size
        
        a = feats[left_start:left_end]
        b = feats[left_end:right_end]
        sim = compute_window_similarity(a, b, similarity_method)
        
        similarities.append(float(sim))
        positions.append(left_end)
        
        if sim < threshold:
            mid = left_end
            if (not boundaries) or (mid - boundaries[-1] >= min_segment_frames):
                boundaries.append(int(mid))
                pos = mid
                left_len = left_init
            else:
                left_len = min(left_len + grow, max_left)
                if (pos + left_len + right_size > T) or (left_len >= max_left):
                    pos += step_reset
                    left_len = left_init
        else:
            left_len = min(left_len + grow, max_left)
            if (pos + left_len + right_size > T) or (left_len >= max_left):
                pos += step_reset
                left_len = left_init
    
    return boundaries, similarities, positions


def traditional_diff_peaks(feats_scaled, fps):
    """Traditional differential peak detection (85th percentile), pre-filtering interval 0.3s"""
    div_boundaries = []
    try:
        from scipy.signal import savgol_filter
        diffs = feats_scaled[1:] - feats_scaled[:-1]
        l2 = np.linalg.norm(diffs, axis=1)
        if len(l2) >= 5:
            win = min(7, len(l2) // 4)
            win += 1 if win % 2 == 0 else 0
            if win >= 3:
                l2_s = savgol_filter(l2, win, 1, mode='mirror')
                thr = np.percentile(l2_s, 85)
                cand = np.where(l2_s > thr)[0]
                gap = max(1, int(0.3 * fps))  # 0.3s
                for c in cand:
                    if not div_boundaries or c - div_boundaries[-1] >= gap:
                        div_boundaries.append(int(c))
    except Exception:
        pass
    return div_boundaries


def refine_boundary(b, feats, fps, radius=None):
    """Refine boundaries: select the local maximum of the smoothed L2 difference curve within ±0.3s"""
    if radius is None:
        radius = max(2, int(0.3 * fps))  # 0.3s
    L = feats.shape[0]
    start = max(2, b - radius)
    end = min(L - 2, b + radius)
    diffs = feats[1:] - feats[:-1]
    l2 = np.linalg.norm(diffs, axis=1)
    l2_smooth = np.convolve(l2, np.ones(3)/3, mode='same') if len(l2) >= 3 else l2
    idx = np.argmax(l2_smooth[start:end+1])
    return int(start + idx)


def post_fuse_and_refine(boundaries_a, boundaries_b, feats_scaled, fps):
    """Fuse two sets of candidates, use 0.3s minimum interval for deduplication, then refine (same 0.3s minimum interval)"""
    MIN_DIST = max(1, int(0.3 * fps))  # 0.3s
    merged = sorted(set(boundaries_a + boundaries_b))
    final = []
    for b in merged:
        if not final or b - final[-1] >= MIN_DIST:
            final.append(int(b))
    # Refine
    refined = []
    for b in final:
        rb = refine_boundary(b, feats_scaled, fps)
        if not refined or rb - refined[-1] >= MIN_DIST:
            refined.append(rb)
    return refined


def evaluate_segmentation_quality(boundaries, total_frames, fps=10):
    """Evaluate segmentation quality"""
    if len(boundaries) == 0:
        return 0.7 if total_frames >= fps * 5 else 0.5
    
    seg_lens, prev = [], 0
    for b in boundaries + [total_frames]:
        seg_lens.append(b - prev)
        prev = b
    
    durs = [l / fps for l in seg_lens]
    mn, av, n, sd = min(durs), np.mean(durs), len(durs), np.std(durs)
    
    score = 1.0
    if mn < 1.0:
        score *= 0.2
    elif mn < 1.5:
        score *= 0.7
    
    if n > 6:
        score *= 0.3
    elif n > 4:
        score *= 0.8
    
    if av < 1.5:
        score *= 0.4
    elif av > 6.0:
        score *= 0.7
    
    if sd > av * 0.8:
        score *= 0.8
    
    return score


def adaptive_asymmetric_search(feats_scaled, fps):
    """Adaptive asymmetric sliding window parameter search"""
    # More dense grid: windows 10/12/15; thresholds 0.72/0.74/0.76/0.80
    window_sizes = [10, 12, 15]
    thresholds = [0.72, 0.74, 0.76, 0.80]
    methods = ['cosine']
    
    best_boundaries, best_score, best_params = [], 0.0, {}
    
    print(f"\n🔍 Adaptive search space:")
    print(f"   Window sizes: {window_sizes} frames = {[w/fps for w in window_sizes]} seconds")
    print(f"   Similarity thresholds: {thresholds}")
    print(f"   Similarity methods: {methods}")
    
    for ws in window_sizes:
        for thr in thresholds:
            for m in methods:
                try:
                    min_seg = max(fps, ws)
                    boundaries, sims, pos = sw_asymmetric(
                        feats_scaled, window_size=ws, step_reset=1,
                        threshold=thr, similarity_method=m, min_segment_frames=min_seg
                    )
                    score = evaluate_segmentation_quality(boundaries, feats_scaled.shape[0], fps)
                    
                    if score > best_score:
                        best_boundaries, best_score = boundaries, score
                        best_params = {
                            'window_size': ws,
                            'threshold': thr,
                            'method': m,
                            'num_boundaries': len(boundaries),
                            'score': score
                        }
                except Exception:
                    continue
    
    return best_boundaries, best_params


def enhanced_runner(feats, fps):
    """Complete Enhanced Segmentation Actuator (Asymmetric Window + Differential Peak + Fusion + Refinement)"""
    print("\n" + "-"*70)
    
    # Standardization Features
    scaler = StandardScaler()
    feats_scaled = scaler.fit_transform(feats)
    
    # 1) Asymmetric sliding window
    print("🔹 Step 1: Asymmetric Sliding Window Segmentation...")
    sw_boundaries, sw_params = adaptive_asymmetric_search(feats_scaled, fps)
    print(f"   ✓ Found {len(sw_boundaries)} boundaries")
    print(f"   ✓ Best parameters: ws={sw_params['window_size']}, thr={sw_params['threshold']:.2f}")
    print(f"   ✓ Quality score: {sw_params['score']:.3f}")
    
    # 2) 传统差分峰值检测（85分位），预去重间隔 0.3s
    print("\n🔹 Step 2: Differential Peak Detection...")
    div_boundaries = traditional_diff_peaks(feats_scaled, fps)
    print(f"   ✓ Found {len(div_boundaries)} boundaries")
    
    # 3) 融合 + 精修
    print("\n🔹 Step 3: Fusion and Refinement...")
    final = post_fuse_and_refine(sw_boundaries, div_boundaries, feats_scaled, fps)
    print(f"   ✓ After fusion: {len(final)} boundaries")
    
    # 4) 质量兜底
    score = evaluate_segmentation_quality(final, feats.shape[0], fps)
    if score < 0.5:
        if len(sw_boundaries) <= len(div_boundaries):
            final = sw_boundaries
            method_used = 'asymmetric_only'
        else:
            final = div_boundaries
            method_used = 'divergence_only'
    else:
        method_used = 'asymmetric_hybrid_div'
    
    # 统计信息
    num_segments = len(final) + 1
    avg_duration = feats.shape[0] / num_segments / fps
    
    print(f"\n📊 Final Statistics:")
    print(f"   Number of Boundaries: {len(final)}")
    print(f"   Number of Segments: {num_segments}")
    print(f"   Average Segment Length: {avg_duration:.1f} seconds")
    print(f"   Final Quality Score: {score:.3f}")
    print(f"   使用方法: {method_used}")
    print("-"*70)
    
    # 构建参数字典
    params = {
        **sw_params,
        'method_used': method_used,
        'sw_boundaries': len(sw_boundaries),
        'div_boundaries': len(div_boundaries),
        'final_boundaries': len(final),
        'final_score': score,
        'num_segments': num_segments,
        'avg_segment_sec': avg_duration
    }
    
    return final, params

def create_segments(boundaries, T):
    """从边界列表创建片段列表"""
    segs = []
    prev = 0
    for b in sorted(boundaries):
        b = int(b)
        if b > prev:
            segs.append([int(prev), int(b)])
            prev = b
    if prev < T:
        segs.append([int(prev), int(T)])
    return segs

def extract_atomic_features(features, boundaries, fps):
    """从CLIP特征中提取每个原子单元的代表性特征"""
    T = features.shape[0]
    segments = []
    
    # 构建片段列表
    prev = 0
    for b in boundaries:
        if b > prev:
            segments.append((prev, b))
            prev = b
    if prev < T:
        segments.append((prev, T))
    
    atomic_features = []
    atomic_centers = []
    
    for start, end in segments:
        # 计算片段的平均特征
        segment_feats = features[start:end]
        mean_feat = np.mean(segment_feats, axis=0)
        
        # L2归一化
        mean_feat = mean_feat / (np.linalg.norm(mean_feat) + 1e-8)
        atomic_features.append(mean_feat)
        
        # 计算时间中心（秒）
        center_time = (start + end) / 2.0 / fps
        atomic_centers.append(center_time)
    
    return np.array(atomic_features), np.array(atomic_centers)

def build_semantic_consistency_map(atomic_features, atomic_centers, sigma):
    """构建语义一致性图谱 (SCM)"""
    K = len(atomic_features)
    M = np.zeros((K, K))
    
    for i in range(K):
        for j in range(K):
            # 1. 语义核：余弦相似度
            K_sem = np.dot(atomic_features[i], atomic_features[j])
            
            # 2. 时间核：高斯RBF
            time_diff = atomic_centers[i] - atomic_centers[j]
            K_temp = np.exp(- (time_diff ** 2) / (2 * sigma ** 2))
            
            # 3. 混合核
            M[i, j] = K_sem * K_temp
    
    return M

def similarity_to_distance(similarity_matrix):
    """将相似度矩阵转换为距离矩阵"""
    sim_clipped = np.clip(similarity_matrix, 0, 1)
    distance = np.sqrt(2 * (1 - sim_clipped))
    np.fill_diagonal(distance, 0)
    return distance

def time_to_frame(time_sec, fps=FPS):
    """将时间（秒）转换为帧索引"""
    return int(time_sec * fps)

def calculate_split_error(actual_splits, target_splits, tolerance=15):
    """计算实际分割点与目标分割点的误差（允许数量不完全匹配）"""
    if len(actual_splits) == 0 or len(target_splits) == 0:
        return float('inf')
    
    # 允许数量有较大差异（放宽到5）
    if abs(len(actual_splits) - len(target_splits)) > 5:
        return float('inf')
    
    # 使用最近邻匹配
    total_error = 0
    matched_count = 0
    
    for target in target_splits:
        # 找到最接近的实际分割点
        min_dist = float('inf')
        for actual in actual_splits:
            dist = abs(actual - target)
            if dist < min_dist:
                min_dist = dist
        
        if min_dist <= tolerance:  # 在容差范围内
            total_error += min_dist
            matched_count += 1
    
    if matched_count == 0:
        return float('inf')
    
    # 惩罚未匹配的分割点
    penalty = abs(len(actual_splits) - len(target_splits)) * 10
    return (total_error / matched_count) + penalty

def get_split_times_from_segments(segments, fps=FPS):
    """从分割结果中提取分割时间点（秒）"""
    split_times = []
    for i in range(len(segments) - 1):
        split_frame = segments[i][1]  # 当前片段的结束帧
        split_time = split_frame / fps
        split_times.append(int(split_time))
    return split_times

def format_time(seconds):
    return f"{int(seconds//60):02d}:{int(seconds%60):02d}"

def labels_to_segments(labels, atomic_segs):
    segments = []
    K = len(labels)
    event_boundaries = [0]
    for i in range(1, K):
        if labels[i] != labels[i-1]:
            event_boundaries.append(i)
    event_boundaries.append(K)
    for i in range(len(event_boundaries) - 1):
        start_idx = event_boundaries[i]
        end_idx = event_boundaries[i+1] - 1
        start_frame = atomic_segs[start_idx][0]
        end_frame = atomic_segs[end_idx][1]
        segments.append((start_frame, end_frame))
    return segments

def calculate_split_error(actual_splits, target_splits, tolerance=15):
    """计算实际分割点与目标分割点的误差"""
    if len(actual_splits) == 0 or len(target_splits) == 0:
        return float('inf')
    
    if abs(len(actual_splits) - len(target_splits)) > 5:
        return float('inf')
    
    total_error = 0
    matched_count = 0
    
    for target in target_splits:
        min_dist = float('inf')
        for actual in actual_splits:
            dist = abs(actual - target)
            if dist < min_dist:
                min_dist = dist
        
        if min_dist <= tolerance:
            total_error += min_dist
            matched_count += 1
    
    if matched_count == 0:
        return float('inf')
    
    penalty = abs(len(actual_splits) - len(target_splits)) * 10
    return (total_error / matched_count) + penalty

In [ ]:
# ## Step 1: Frame Extraction
print("\n" + "="*70)
print("[Step 1/3] Frame Extraction")
print("="*70)

def extract_frames(video_path):
    video_files = sorted([f for f in os.listdir(video_path) if f.endswith(('.mp4', '.avi', '.mov'))])
    print(f"🎬 Found {len(video_files)} video(s) in '{video_path}': {video_files}")
    for video_file in tqdm(video_files, desc="Extracting frames"):
        vid_path = Path(path.join(video_path, video_file))
        video_name = vid_path.stem
        temp_path = video_path + f"_{FPS}FPS"
        output_dir = Path(OUTPUT_DIR) / temp_path / video_name
        output_dir.mkdir(parents=True, exist_ok=True)
        if CLEAN_FRAMES:
            # print("Clear old frames...")
            for p in output_dir.glob('*.jpg'):
                p.unlink()

        frame_files = sorted(list(output_dir.glob('*.jpg')))
        if frame_files and REUSE_EXISTING_FRAMES:
            # print(f"✅ Reusing existing {len(frame_files)} frames")
            pass
        else:
            # print("Using FFmpeg to extract frames...")
            cmd = [
                'ffmpeg', '-y', '-v', 'error',
                '-i', str(vid_path),
                '-vf', f'fps={FPS}',
                '-q:v', '2',
                str(output_dir / '%06d.jpg')
            ]
            subprocess.run(cmd, check=True)
            frame_files = sorted(list(output_dir.glob('*.jpg')))

extract_frames(VIDEO_PATH)

In [ ]:
# ## Step 2: CLIP encoding
print("\n" + "="*70)
print("[Step 2/3] CLIP encoding")
print("="*70)

print("Load the CLIP model...")

model, preprocess = clip.load("ViT-B/32", device=device)


def get_filenames(dir_path):
    return [f for f in os.listdir(dir_path)]

frame_filenames = sorted(get_filenames(path.join(OUTPUT_DIR, VIDEO_PATH + f"_{FPS}FPS")))

for f in tqdm(frame_filenames, desc="Processing videos"):
    images = []
    frame_files = get_filenames(path.join(OUTPUT_DIR, VIDEO_PATH + f'_{FPS}FPS', f))
    for ff in tqdm(frame_files, desc=f"Processing frames for {f}", leave=False):
        try:
            with Image.open(path.join(OUTPUT_DIR, VIDEO_PATH + f'_{FPS}FPS', f, ff)) as img:
                images.append(preprocess(img.convert("RGB")).to(device))
        except Exception as e:
            print(f"⚠️  Skip corrupted frame: {ff}")
            continue
    
    # make images from list to tensor
    images = torch.stack(images)
    output_dir = Path(PREPROCESSED_DIR) / (VIDEO_PATH + f"_{FPS}FPS")
    output_dir.mkdir(parents=True, exist_ok=True)
    torch.save(images, path.join(output_dir, f"{f}.pt"))



# images = []
# for f in tqdm(frame_files, desc="Preprocessing frames"):
#     try:
#         with Image.open(f) as img:
#             images.append(preprocess(img).to(device))
#     except Exception as e:
#         print(f"⚠️  Skip corrupted frames: {f}")
# feats_list = []



# with torch.no_grad():
#     for i in tqdm(range(0, num_frames, BATCH_SIZE), desc="Encoding frame"):
#         images = []
#         for f in frame_files[i:i+BATCH_SIZE]:
#             try:
#                 with Image.open(f) as img:
#                     images.append(preprocess(img.convert("RGB")))
#             except Exception as e:
#                 print(f"⚠️  Skip corrupted frames: {f}")
#                 continue
        
#         if images:
#             image_input = torch.stack(images).to(device)
#             features = model.encode_image(image_input)
#             features = features / features.norm(dim=-1, keepdim=True)
#             feats_list.append(features.cpu().numpy())

# features = np.concatenate(feats_list, axis=0)
# print(f"✅ Feature matrix: {features.shape}")

In [6]:
def get_filenames(dir_path):
    return [f for f in os.listdir(dir_path)]

torch_files = get_filenames(path.join(PREPROCESSED_DIR, VIDEO_PATH + f"_{FPS}FPS"))

In [9]:
model, preprocess = clip.load("ViT-B/32", device=device)
with torch.no_grad():
    for f in tqdm(torch_files, desc="Encoding frames"):
        images = torch.load(path.join(PREPROCESSED_DIR, VIDEO_PATH + f"_{FPS}FPS", f))
        image_features = model.encode_image(images.to(device))

Encoding frames:   0%|          | 0/1570 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
image_features.shape

In [ ]:
print("✅ Segmentation algorithm loaded (Enhanced version: Asymmetric Window + Differential Peak + Fusion + Refinement)")

# ## Step 3: Execute Atomic Segmentation


print("\n" + "="*70)
print("[Step 3/3] Atomic-level Event Segmentation")
print("="*70)

# Perform a complete enhanced segmentation (asymmetric windowing + differential peak detection + fusion + refinement)）
boundaries, params = enhanced_runner(features, FPS)

results = {'asymmetric': {'boundaries': boundaries, 'params': params}}

print("\n✅ Atomic segmentation completed")

# ## Save the atomic segmentation results

T = features.shape[0]
segs = create_segments(boundaries, T)

json_data = {
    'video_name': video_name,
    'num_frames': int(T),
    'fps': float(FPS),
    'video_duration': round(T / FPS, 3),
    'num_boundaries': len(boundaries),
    'num_segments': len(segs),
    'boundaries_frame_indices': [int(b) for b in boundaries],
    'boundaries_timestamps': [round(b / FPS, 3) for b in boundaries],
    'segments_timestamps': [[round(s[0]/FPS, 3), round(s[1]/FPS, 3)] for s in segs],
    'method': 'enhanced_asymmetric_hybrid',
    'parameters': params
}

out_path = output_dir / f"{video_name}_atomic_segmentation.json"
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(json_data, f, ensure_ascii=False, indent=2)

print(f"✅ Atomic segmentation JSON: {out_path}")

# Visualize atomic segmentation
fig, ax = plt.subplots(figsize=(20, 3))
colors = plt.cm.Set3(np.linspace(0, 1, len(segs)))

for idx, (start, end) in enumerate(segs):
    ax.barh(0, end-start, left=start/FPS, height=0.8, 
            color=colors[idx % len(colors)], edgecolor='black', linewidth=0.5)
    # Add a timestamp (mm:ss-mm:ss)
    seg_center = (start + end) / 2 / FPS
    start_time = start / FPS
    end_time = end / FPS
    
    start_min = int(start_time // 60)
    start_sec = int(start_time % 60)
    end_min = int(end_time // 60)
    end_sec = int(end_time % 60)
    
    time_label = f'{start_min:02d}:{start_sec:02d}-{end_min:02d}:{end_sec:02d}'
    
    if (end - start) / FPS > 2:  # Only display time on longer segments
        ax.text(seg_center, 0, time_label, ha='center', va='center', 
                fontsize=6, color='white', fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.6))

ax.set_xlim(0, T/FPS)
ax.set_ylim(-0.5, 0.5)
ax.set_yticks([])
ax.set_xlabel('Time (seconds)', fontsize=12)
ax.set_title(f'Atomic-level Segmentation - {len(segs)} Segments', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
atomic_viz_path = output_dir / f"{video_name}_atomic_segments.png"
plt.savefig(atomic_viz_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ Atomic segmentation visualization: {atomic_viz_path}")

In [ ]:
# ## Phase I: Building a Semantic Consistency Map


print("\n" + "="*70)
print("Phase I: 构建语义一致性图谱 (SCM)")
print("="*70)

K = len(boundaries) + 1  # Number of atomic units
print(f"Number of atomic units: {K}")





# Extract atomic unit features
atomic_features, atomic_centers = extract_atomic_features(features, boundaries, FPS)

print(f"Atomic feature matrix: {atomic_features.shape}")
print(f"Time center range: {atomic_centers[0]:.2f}s ~ {atomic_centers[-1]:.2f}s")





# Building an SCM
sigma = HIERARCHY_CONFIG['sigma']
SCM = build_semantic_consistency_map(atomic_features, atomic_centers, sigma)

print(f"\n✅ SCM building completed")
print(f"   Matrix size: {SCM.shape}")
print(f"   Time-resolved bandwidth σ: {sigma}秒")
print(f"   Affinity range: [{SCM.min():.4f}, {SCM.max():.4f}]")


# Visualize SCM
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

im1 = axes[0].imshow(SCM, cmap='hot', aspect='auto', interpolation='nearest')
axes[0].set_title(
    f'Semantic Consistency Map\nσ={sigma}s, {K} Atomic Units',
    fontsize=12, fontweight='bold'
)
axes[0].set_xlabel('Atomic Unit Index', fontsize=10)
axes[0].set_ylabel('Atomic Unit Index', fontsize=10)
plt.colorbar(im1, ax=axes[0], label='Affinity Score')

band_width = min(15, K//2)
band_mask = np.abs(np.arange(K)[:, None] - np.arange(K)[None, :]) <= band_width
SCM_band = SCM.copy()
SCM_band[~band_mask] = 0

im2 = axes[1].imshow(SCM_band, cmap='hot', aspect='auto', interpolation='nearest')
axes[1].set_title(
    f'Band Diagonal Structure (±{band_width} Units)',
    fontsize=12, fontweight='bold'
)
axes[1].set_xlabel('Atomic Unit Index', fontsize=10)
axes[1].set_ylabel('Atomic Unit Index', fontsize=10)
plt.colorbar(im2, ax=axes[1], label='Affinity Score')

plt.tight_layout()
scm_path = output_dir / f"{video_name}_SCM.png"
plt.savefig(scm_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ SCM visualization: {scm_path}")

In [ ]:

# ## Phase II: Analysis of Hierarchical Topology (Ward's HAC)


print("\n" + "="*70)
print("Phase II: Analysis of Hierarchical Topology (Ward's HAC)")
print("="*70)





distance_matrix = similarity_to_distance(SCM)
condensed_distance = squareform(distance_matrix, checks=False)

# Ward's Hierarchical Clustering
linkage_matrix = linkage(condensed_distance, method='ward')
merge_distances = linkage_matrix[:, 2]

print(f"✅ Hierarchical clustering completed")
print(f"   Merge distance range: [{merge_distances.min():.4f}, {merge_distances.max():.4f}]")


# Visualize the dendrogram
fig, ax = plt.subplots(figsize=(18, 8))
dendrogram_data = dendrogram(
    linkage_matrix,
    ax=ax,
    color_threshold=np.percentile(merge_distances, 85),
    above_threshold_color='gray',
    leaf_font_size=6
)
ax.set_title(
    f'Hierarchical Dendrogram - {video_name} ({K} Atomic Units)',
    fontsize=14, fontweight='bold'
)
ax.set_xlabel('Atomic Unit Index', fontsize=12)
ax.set_ylabel('Ward Distance', fontsize=12)
ax.grid(True, alpha=0.3, axis='y')

# Add a divider
cut_levels = HIERARCHY_CONFIG['cut_levels']
for level_name, percentile in cut_levels.items():
    if percentile is not None:
        threshold = np.percentile(merge_distances, percentile)
        ax.axhline(y=threshold, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
        ax.text(K*0.98, threshold, f'{level_name} ({percentile}%)', 
                verticalalignment='bottom', fontsize=9, color='red')

plt.tight_layout()
dendrogram_path = output_dir / f"{video_name}_dendrogram.png"
plt.savefig(dendrogram_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ Dendrogram: {dendrogram_path}")

# ## Multi-Granularity Segmentation

# %%
print("\n" + "="*70)
print("Multi-Granularity Event Segmentation")
print("="*70)

hierarchical_segmentations = {}
cut_levels = HIERARCHY_CONFIG['cut_levels']

for level_name, percentile in cut_levels.items():
    if percentile is None:
        # No cutting at the atomic level
        hierarchical_segmentations[level_name] = {
            'threshold': 0.0,
            'num_events': K,
            'labels': np.arange(1, K+1)
        }
        print(f"{level_name:12s}: {K:3d} an atomic unit")
    else:
        threshold = np.percentile(merge_distances, percentile)
        cluster_labels = fcluster(linkage_matrix, threshold, criterion='distance')
        num_clusters = len(np.unique(cluster_labels))
        
        hierarchical_segmentations[level_name] = {
            'threshold': threshold,
            'num_events': num_clusters,
            'labels': cluster_labels
        }
        
        print(f"{level_name:12s}: {num_clusters:3d} events "
              f"(average {K/num_clusters:.1f} atoms/event, threshold={threshold:.3f})")

# %%
# Visualize multi-granularity segmentation
fig, axes = plt.subplots(len(hierarchical_segmentations), 1, 
                         figsize=(20, 2*len(hierarchical_segmentations)))

level_names = ['atomic', 'component', 'side', 'activity']

for idx, (level_name, ax) in enumerate(zip(level_names, axes)):
    level_data = hierarchical_segmentations[level_name]
    labels = level_data['labels']
    num_events = level_data['num_events']
    colors = plt.cm.tab20(np.linspace(0, 1, num_events))
    
    # Visualize each atomic unit
    for i in range(K):
        cluster_id = labels[i] - 1
        color = colors[cluster_id % len(colors)]
        ax.barh(0, 1, left=i, height=0.8, color=color, edgecolor='white', linewidth=0.5)
    
    # Add event boundaries and time annotations (mm:ss-mm:ss)
    event_boundaries = []
    for i in range(1, K):
        if labels[i] != labels[i-1]:
            event_boundaries.append(i)
    
    # Mark the start and end times for each event
    prev_boundary = 0
    for boundary in event_boundaries + [K]:
        event_center_idx = (prev_boundary + boundary) / 2
        
        # Calculate the start and end times of the event (in seconds)
        start_idx = int(prev_boundary) if prev_boundary < len(atomic_centers) else len(atomic_centers) - 1
        end_idx = int(boundary - 1) if boundary - 1 < len(atomic_centers) else len(atomic_centers) - 1
        
        start_time = atomic_centers[start_idx]
        end_time = atomic_centers[end_idx]
        
        # Format as mm:ss
        start_min = int(start_time // 60)
        start_sec = int(start_time % 60)
        end_min = int(end_time // 60)
        end_sec = int(end_time % 60)
        
        time_label = f'{start_min:02d}:{start_sec:02d}-{end_min:02d}:{end_sec:02d}'
        
        # Only display time on larger events
        if (boundary - prev_boundary) > K * 0.03:  # Events account for more than 3%
            ax.text(event_center_idx, 0, time_label, ha='center', va='center',
                    fontsize=7, color='white', fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='black', alpha=0.7))
        
        # Visualize segment boundaries
        if boundary < K:
            ax.axvline(x=boundary, color='black', linestyle='-', linewidth=2, alpha=0.8)
        
        prev_boundary = boundary
    
    ax.set_xlim(0, K)
    ax.set_ylim(-0.5, 0.5)
    ax.set_yticks([])
    ax.set_xlabel('Atomic Unit Index', fontsize=10)
    ax.set_title(
        f'{level_name.upper()} Level - {num_events} Events',
        fontsize=11, fontweight='bold'
    )
    ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
hierarchy_path = output_dir / f"{video_name}_hierarchy.png"
plt.savefig(hierarchy_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Hierarchical Segmentation Visualization: {hierarchy_path}")


# ## Save the hierarchy results

# %%
# Save the hierarchy as JSON
hierarchy_json = {
    'video_name': video_name,
    'num_atomic_units': int(K),
    'scm_sigma': float(sigma),
    'linkage_method': 'ward',
    'hierarchical_levels': {}
}

for level_name, level_data in hierarchical_segmentations.items():
    hierarchy_json['hierarchical_levels'][level_name] = {
        'threshold': float(level_data['threshold']),
        'num_events': int(level_data['num_events']),
        'cluster_labels': level_data['labels'].tolist()
    }

hierarchy_json_path = output_dir / f"{video_name}_hierarchy.json"
with open(hierarchy_json_path, 'w', encoding='utf-8') as f:
    json.dump(hierarchy_json, f, ensure_ascii=False, indent=2)

print(f"✅ Hierarchical Structure JSON: {hierarchy_json_path}")


# ## 📊 Final Summary

# %%
print("\n" + "="*70)
print(f"📊 Results of the toy car disassembly and segmentation - {video_name}")
print("="*70)
print(f"Video length: {T/FPS:.2f} seconds ({T} frames), FPS: {FPS}")
print()

print("[Atomic-Level Segmentation]")
print(f"  Method: Asymmetric Sliding Window")
print(f"  Number of Boundaries: {len(boundaries)}")
print(f"  Number of Segments: {len(segs)}")
print(f"  Average Segment Length: {params['avg_segment_sec']:.1f} seconds")
print(f"  Quality Score: {params['final_score']:.3f}")
print()

print("[Hierarchy]")
for level_name in level_names:
    level_data = hierarchical_segmentations[level_name]
    print(f"  {level_name:12s}: {level_data['num_events']:3d} an event")

print()
print("💡 Explanation:")
print("  - ATOMIC    : Disassemble/assemble a single screw, remove/install a single small component")
print("  - COMPONENT : Disassemble/assemble a single wheel, body, or other complete component")
print("  - SIDE      : Disassemble/assemble the left, right side (e.g., left and right wheel assembly)")
print("  - ACTIVITY  : Top level: Disassembling the toy car vs. Assembling the toy car")
print()
print(f"📁 All outputs saved to: {output_dir}")
print("   - *_atomic_segmentation.json (Atomic Segmentation)")
print("   - *_atomic_segments.png (Atomic Segmentation Visualization)")
print("   - *_hierarchy.json (Hierarchical Structure)")
print("   - *_SCM.png (Semantic Consistency Map)")
print("   - *_dendrogram.png (Dendrogram)")
print("   - *_hierarchy.png (Multi-Granularity Segmentation)")
print()
print("🎉 Pipeline execution completed!")
print("="*70)

In [ ]:
# ======================================================================
# Parameter Back-Testing: Finding the Optimal Parameters Based on Target Breakout Timing
# ======================================================================


print("="*70)
print("🔍 Parameter Back-Testing")
print("="*70)
print()

# Target segmentation time (seconds)
target_splits_top = [148]  # 2:28 - Top-level breakpoints (2 events)
target_splits_lower = [8, 31, 52, 64, 148, 160, 180, 208, 225, 251, 278]  # Lower-level breakpoints

print("🎯 Target Split Times:")
print(f"  Top Level (2 events): {target_splits_top[0]//60:02d}:{target_splits_top[0]%60:02d}")
print(f"  Lower Levels ({len(target_splits_lower)+1} events): ", end="")
for t in target_splits_lower:
    print(f"{t//60:02d}:{t%60:02d}", end=" ")
print("\n")

# Parameter search range (significantly expanded)
ws_range = range(5, 35, 3)  # Sliding window size: 5, 8, 11, ..., 32
thr_range = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]  # Threshold
sigma_range = [10, 20, 30, 40, 50, 60, 80, 100, 120]  # SCM time kernel bandwidth
tolerance = 20  # Allowed time error (seconds) - relaxed to 20 seconds

print("🔧 Parameter Search Range:")
print(f"  Sliding Window Size (ws): {list(ws_range)}")
print(f"  Threshold (thr): {thr_range}")
print(f"  SCM Time Kernel Bandwidth (sigma): {sigma_range}")
print(f"  Tolerance: ±{tolerance} seconds")
print()

# Let's run a quick test to see what results the current parameters produce.
print("🧪 Quick Test of Current Parameters...")
try:
    # Use variables already defined in the notebook
    print(f"  Current atomic segmentation produces {len(segs)} segments")
    print(f"  Number of atomic units: {K}")

    # Recalculate hierarchical clustering (since this is a new cell)
    test_cos = atomic_features @ atomic_features.T
    test_time_diff = np.abs(atomic_centers[:, None] - atomic_centers[None, :])
    test_temporal = np.exp(-test_time_diff**2 / (2 * 50**2))
    test_SCM = test_cos * test_temporal
    test_dist = 1 - test_SCM
    np.fill_diagonal(test_dist, 0)
    test_condensed = test_dist[np.triu_indices(K, k=1)]
    test_Z = linkage(test_condensed, method='ward')
    
    # Test 2 events
    test_labels_2 = fcluster(test_Z, t=2, criterion='maxclust')
    test_splits_2 = [int(atomic_centers[i]) for i in range(1, K) if test_labels_2[i] != test_labels_2[i-1]]
    print(f"  The dividing line between the two events: ", end="")
    for t in test_splits_2:
        print(f"{t//60:02d}:{t%60:02d}", end=" ")
    print()
    
    # Test 12 events
    test_labels_12 = fcluster(test_Z, t=12, criterion='maxclust')
    test_splits_12 = [int(atomic_centers[i]) for i in range(1, K) if test_labels_12[i] != test_labels_12[i-1]]
    print(f"  12 Milestones: ", end="")
    for t in test_splits_12:
        print(f"{t//60:02d}:{t%60:02d}", end=" ")
    print()
    print()
except Exception as e:
    print(f"  Test failed: {e}")
    import traceback
    traceback.print_exc()
    print()



# Prepare normalized features (as in the notebook)
feats_unit = features / (np.linalg.norm(features, axis=1, keepdims=True) + 1e-8)
print(f"✅ Feature normalization complete: {feats_unit.shape}")
print()

# Let's test a few parameter combinations first to see how many segments are generated.
print("🧪 Testing a few parameter combinations...")
scaler_test = StandardScaler()
feats_scaled_test = scaler_test.fit_transform(features)
test_params = [(10, 0.5), (15, 0.6), (20, 0.7), (25, 0.8)]
for ws, thr in test_params:
    try:
        bounds, _, _ = sw_asymmetric(feats_scaled_test, window_size=ws, threshold=thr, 
                                      similarity_method='cosine', min_segment_frames=max(FPS, ws))
        print(f"  ws={ws}, thr={thr:.1f}: {len(bounds)} a boundary -> {len(bounds)+1} a clip")
    except Exception as e:
        print(f"  ws={ws}, thr={thr:.1f}: Error - {e}")
print()

print("🔄 Starting parameter search...")
print()

best_params = None
best_error = float('inf')
best_results = None
all_results = []  # Save all valid results
error_count = 0  # Count the number of anomalies
debug_info = []  # Save debug information

total_combinations = len(ws_range) * len(thr_range) * len(sigma_range)
tested = 0

for ws, thr, sigma in itertools.product(ws_range, thr_range, sigma_range):
    tested += 1
    if tested % 50 == 0:
        print(f"  Tested {tested}/{total_combinations} Group parameters... (Found {len(all_results)} An effective combination)")
    
    try:
        # 1. Atomic partitioning - Using the same complete method as in actual operation
        # Normalize features
        scaler_loop = StandardScaler()
        feats_scaled_loop = scaler_loop.fit_transform(features)
        
        # Step 1: Asymmetric sliding window
        min_seg = max(FPS, ws)
        sw_boundaries, _, _ = sw_asymmetric(
            feats_scaled_loop, window_size=ws, step_reset=1,
            threshold=thr, similarity_method='cosine', min_segment_frames=min_seg
        )
        
        # Step 2: Differential Peak Detection
        div_boundaries = traditional_diff_peaks(feats_scaled_loop, FPS)
        
        # Step 3: Fusion and Refinement (using the same logic as enhanced_runner)
        boundaries_test = post_fuse_and_refine(sw_boundaries, div_boundaries, feats_scaled_loop, FPS)
        
        # Convert to segment format
        segs_test = []
        prev = 0
        for b in boundaries_test:
            segs_test.append((prev, b))
            prev = b + 1
        segs_test.append((prev, features.shape[0] - 1))
        
        if len(segs_test) < 2:
            if tested <= 10:  # Record more
                debug_info.append(f"ws={ws}, thr={thr:.2f}, sigma={sigma}: Too few segments ({len(segs_test)})")
            continue
        
        # Record some successful cases
        if tested <= 10 and len(segs_test) >= 10:
            debug_info.append(f"ws={ws}, thr={thr:.2f}, sigma={sigma}: Successfully generated {len(segs_test)} segments")
        
        # 2. Build atomic features and centers
        atomic_features_test = []
        atomic_centers_test = []
        for start, end in segs_test:
            seg_feat = features[start:end+1].mean(axis=0)
            atomic_features_test.append(seg_feat)
            atomic_centers_test.append((start + end) / 2 / FPS)
        
        atomic_features_test = np.array(atomic_features_test)
        atomic_centers_test = np.array(atomic_centers_test)
        K_test = len(segs_test)
        
        # 3. Build SCM
        cos_sim_test = atomic_features_test @ atomic_features_test.T
        time_diff_test = np.abs(atomic_centers_test[:, None] - atomic_centers_test[None, :])
        temporal_kernel_test = np.exp(-time_diff_test**2 / (2 * sigma**2))
        SCM_test = cos_sim_test * temporal_kernel_test
        
        # 4. Hierarchical Clustering
        dist_matrix_test = 1 - SCM_test
        np.fill_diagonal(dist_matrix_test, 0)
        condensed_dist_test = dist_matrix_test[np.triu_indices(K_test, k=1)]
        Z_test = linkage(condensed_dist_test, method='ward')
        
        # Test top level (2 events)
        labels_top = fcluster(Z_test, t=2, criterion='maxclust')
        
        # Find top level split points
        top_splits = []
        for i in range(1, K_test):
            if labels_top[i] != labels_top[i-1]:
                split_time = int(atomic_centers_test[i])
                top_splits.append(split_time)
        
        # Calculate top level error
        top_error = calculate_split_error(top_splits, target_splits_top, tolerance)
        
        # Test multiple lower level event counts (allowing a certain range)
        best_lower_error = float('inf')
        best_lower_splits = []
        best_num_events = 0
        
        for num_events_lower in range(8, 16):  # Try 8-15 events
            try:
                labels_lower = fcluster(Z_test, t=num_events_lower, criterion='maxclust')
                
                # Find lower level split points
                lower_splits = []
                for i in range(1, K_test):
                    if labels_lower[i] != labels_lower[i-1]:
                        split_time = int(atomic_centers_test[i])
                        lower_splits.append(split_time)
                
                # Calculate lower level error
                lower_error = calculate_split_error(lower_splits, target_splits_lower, tolerance)
                
                if lower_error < best_lower_error:
                    best_lower_error = lower_error
                    best_lower_splits = lower_splits
                    best_num_events = num_events_lower
            except:
                continue
        
        if best_lower_error == float('inf'):
            if len(all_results) < 5:  # Record why it was filtered
                debug_info.append(f"ws={ws}, thr={thr:.2f}, sigma={sigma}: 下层误差过大 (top_error={top_error:.1f})")
            continue
        
        lower_error = best_lower_error
        lower_splits = best_lower_splits
        
        # Record found valid combinations
        if len(all_results) < 5:
            debug_info.append(f"ws={ws}, thr={thr:.2f}, sigma={sigma}: 找到有效组合! top_err={top_error:.1f}, lower_err={lower_error:.1f}")
        
        # Total error (weighted)
        total_error = top_error * 3 + lower_error  # Top-level error has higher weight
        
        # Save all valid results
        all_results.append({
            'params': {'ws': ws, 'thr': thr, 'sigma': sigma},
            'top_splits': top_splits,
            'lower_splits': lower_splits,
            'top_error': top_error,
            'lower_error': lower_error,
            'total_error': total_error,
            'num_atomic': K_test,
            'num_events_lower': best_num_events
        })
        
        if total_error < best_error:
            best_error = total_error
            best_params = {'ws': ws, 'thr': thr, 'sigma': sigma}
            best_results = {
                'top_splits': top_splits,
                'lower_splits': lower_splits,
                'top_error': top_error,
                'lower_error': lower_error,
                'num_atomic': K_test
            }
    
    except Exception as e:
        error_count += 1
        if error_count <= 5:  # Only record the first 5 errors
            debug_info.append(f"ws={ws}, thr={thr:.2f}, sigma={sigma}: Exception - {str(e)[:50]}")
        continue

print()
print("="*70)
print("✅ Parameter search complete!")
print("="*70)
print()

print(f"📊 Statistics:")
print(f"  Number of test combinations: {tested}")
print(f"  Exception count: {error_count}")
print(f"  Valid combinations: {len(all_results)}")
print()

if debug_info:
    print("🔍 Debug information (first few):")
    for info in debug_info[:10]:
        print(f"  {info}")
    print()

if best_params is not None:
    print("🎯 Best parameters:")
    print(f"  Sliding window size (ws): {best_params['ws']}")
    print(f"  Threshold (thr): {best_params['thr']:.2f}")
    print(f"  SCM time kernel bandwidth (sigma): {best_params['sigma']} seconds")
    print(f"  Number of atomic units: {best_results['num_atomic']}")
    print()
    
    print("📊 Split results comparison:")
    print()
    print("  Top level (2 events):")
    print(f"    Target split point: {target_splits_top[0]//60:02d}:{target_splits_top[0]%60:02d}")
    if best_results['top_splits']:
        for t in best_results['top_splits']:
            print(f"    Actual split point: {t//60:02d}:{t%60:02d}")
        print(f"    Error: {best_results['top_error']:.1f} seconds")
    print()
    
    print(f"  Lower level ({len(best_results['lower_splits'])+1} events):")
    print("    Target split points: ", end="")
    for t in target_splits_lower:
        print(f"{t//60:02d}:{t%60:02d}", end=" ")
    print()
    print("    Actual split points: ", end="")
    for t in best_results['lower_splits']:
        print(f"{t//60:02d}:{t%60:02d}", end=" ")
    print()
    print(f"    Average error: {best_results['lower_error']:.1f} seconds")
    print()
    
    print("💡 Suggestions:")
    print(f"  Modify the parameters in the notebook to:")
    print(f"    - Sliding window: ws={best_params['ws']}, thr={best_params['thr']:.2f}")
    print(f"    - SCM: sigma={best_params['sigma']}")
    print(f"    - Hierarchical clustering: Top level with 2 events, lower level with approximately {len(best_results['lower_splits'])+1} events")
else:
    print("❌ No fully matching parameter combination found")
    print()
    
    # Show the top 10 most relevant results
    if all_results:
        print("📋 Most similar parameter combinations (sorted by error):")
        print()
        all_results_sorted = sorted(all_results, key=lambda x: x['total_error'])
        
        for i, result in enumerate(all_results_sorted[:10], 1):
            params = result['params']
            print(f"  {i}. ws={params['ws']}, thr={params['thr']:.2f}, sigma={params['sigma']}")
            print(f"     Atomic units: {result['num_atomic']}个, Lower level events: {result['num_events_lower']}个")
            print(f"     Top level error: {result['top_error']:.1f} seconds, Lower level error: {result['lower_error']:.1f} seconds")
            print(f"     Top level splits: ", end="")
            for t in result['top_splits']:
                print(f"{t//60:02d}:{t%60:02d}", end=" ")
            print(f" (Objective: {target_splits_top[0]//60:02d}:{target_splits_top[0]%60:02d})")
            print(f"     Lower level splits: ", end="")
            for t in result['lower_splits']:
                print(f"{t//60:02d}:{t%60:02d}", end=" ")
            print()
            print()
    else:
        print("   No valid parameter combinations found")
        print("   Suggestions:")
        print("   1. Further expand the search range")
        print("   2. Check if the target split points match the video content")
        print("   3. Adjust the tolerance of the target split points")

In [ ]:
# ======================================================================
# Save the partitioning results with the optimal parameters (using fixed values of ws=14, thr=0.50, and sigma=80)
# ======================================================================

print("="*70)
print("💾 Save the segmentation results with the optimal parameters")
print("="*70)
print()

# The optimal parameters identified using an improved back-testing method (the actual boundary is at 02:29)
ws = 23
thr = 0.80
sigma = 100
print(f"📋 Optimal parameters (matching actual boundaries): ws={ws}, thr={thr:.2f}, sigma={sigma}")
print("📋 The actual boundaries generated by this set of parameters are at 02:29 (very close to the objective of 02:28!)")
print()

# Run again to obtain the complete data (a method identical to the back-testing with parameters)
scaler_save = StandardScaler()
feats_scaled_save = scaler_save.fit_transform(features)

min_seg = max(FPS, ws)
sw_boundaries_save, _, _ = sw_asymmetric(
    feats_scaled_save, window_size=ws, step_reset=1,
    threshold=thr, similarity_method='cosine', min_segment_frames=min_seg
)

div_boundaries_save = traditional_diff_peaks(feats_scaled_save, FPS)
boundaries_save = post_fuse_and_refine(sw_boundaries_save, div_boundaries_save, feats_scaled_save, FPS)

# Convert to segment format (consistent with parameter back-testing)
segs_save = []
prev = 0
for b in boundaries_save:
    segs_save.append((prev, b))
    prev = b + 1
segs_save.append((prev, features.shape[0] - 1))

print(f"✓ Atomic segments: {len(segs_save)} segments")

# Build atomic features and centers (consistent with parameter back-testing)
atomic_features_save = []
atomic_centers_save = []
for start, end in segs_save:
    seg_feat = features[start:end+1].mean(axis=0)  # closed interval
    atomic_features_save.append(seg_feat)  # not normalized
    atomic_centers_save.append((start + end) / 2 / FPS)

atomic_features_save = np.array(atomic_features_save)
atomic_centers_save = np.array(atomic_centers_save)
K_save = len(segs_save)

print(f"✓ Atomic features: {atomic_features_save.shape}")

# Build SCM (consistent with parameter back-testing)
cos_sim_save = atomic_features_save @ atomic_features_save.T
time_diff_save = np.abs(atomic_centers_save[:, None] - atomic_centers_save[None, :])
temporal_kernel_save = np.exp(-time_diff_save**2 / (2 * sigma**2))
SCM_save = cos_sim_save * temporal_kernel_save

print(f"✓ SCM: {SCM_save.shape}")

# Hierarchical Clustering (consistent with parameter back-testing)
dist_matrix_save = 1 - SCM_save
np.fill_diagonal(dist_matrix_save, 0)
condensed_dist_save = dist_matrix_save[np.triu_indices(K_save, k=1)]
Z_save = linkage(condensed_dist_save, method='ward')

print(f"✓ Clustering Matrix: {Z_save.shape}")
print()

# Verify Boundary Points
labels_verify = fcluster(Z_save, t=2, criterion='maxclust')
verify_splits = []
for i in range(1, K_save):
    if labels_verify[i] != labels_verify[i-1]:
        split_time = int(atomic_centers_save[i])
        verify_splits.append(split_time)

print("🔍 Verify Boundary Points (Atomic Centers):")
for t in verify_splits:
    print(f"  {t//60:02d}:{t%60:02d} ({t} seconds)")
print()

# Save all results
np.save('optimal_segs.npy', np.array(segs_save, dtype=object))
np.save('optimal_atomic_features.npy', atomic_features_save)
np.save('optimal_atomic_centers.npy', atomic_centers_save)
np.save('optimal_SCM.npy', SCM_save)
np.save('optimal_linkage.npy', Z_save)
np.save('optimal_params.npy', np.array([ws, thr, sigma]))

print("="*70)
print("✅ Save completed!")
print("="*70)
print()
print("📁 Saved files:")
print("  • optimal_segs.npy - Atomic segments")
print("  • optimal_atomic_features.npy - Atomic features")
print("  • optimal_atomic_centers.npy - Atomic centers")
print("  • optimal_SCM.npy - Semantic Consistency Matrix")
print("  • optimal_linkage.npy - Clustering Matrix")
print("  • optimal_params.npy - Parameters")
print()

# Output detailed segmentation timestamps
print("="*70)
print("📋 Detailed breakdown of time points")
print("="*70)
print()
    


# Generate multi-level segmentation
hierarchical_segs = {}
hierarchical_segs['activity'] = labels_to_segments(fcluster(Z_save, t=2, criterion='maxclust'), segs_save)
hierarchical_segs['side'] = labels_to_segments(fcluster(Z_save, t=4, criterion='maxclust'), segs_save)
hierarchical_segs['component'] = labels_to_segments(fcluster(Z_save, t=8, criterion='maxclust'), segs_save)

for level_name, icon, title in zip(
    ['activity', 'side', 'component'],
    ['🔴', '🟠', '🟢'],
    ['ACTIVITY', 'SIDE', 'COMPONENT']
):
    segs_level = hierarchical_segs[level_name]
    num_events = len(segs_level)
    print(f"{icon} {title}floor - {num_events} an event")
    print("="*70)
    for i, (start_frame, end_frame) in enumerate(segs_level):
        start_time = start_frame / FPS
        end_time = end_frame / FPS
        duration_event = end_time - start_time
        print(f"  Event {i+1}: {format_time(start_time)} - {format_time(end_time)}  "
                f"(Duration: {duration_event:6.1f} seconds, {int(duration_event/60):2d} mins {int(duration_event%60):02d} secs)")
    if num_events > 1:
        boundary_times = [seg[1] / FPS for seg in segs_level[:-1]]
        print(f"\n  🔹 Boundary Points ({len(boundary_times)}): {', '.join([format_time(t) for t in boundary_times])}")
    print("\n")

    print("="*70)
    print("💡 Now you can load these results in the visualization notebook")
    print()

In [ ]:
# ======================================================================
# Parameter Backtracking Test (Improved Version): Matching Actual Boundary Frames Instead of Atomic Center Points
# ======================================================================

import itertools
from scipy.cluster.hierarchy import linkage, fcluster

print("="*70)
print("🔍 Parameter Backtracking Test (Improved Version - Matching Actual Boundary Frames)")
print("="*70)
print()

# Target segmentation time points (seconds) - this time we want to match actual boundaries
target_splits_top = [148]  # 2:28 - top-level split points (2 events)
target_splits_lower = [8, 31, 52, 64, 148, 160, 180, 208, 225, 251, 278]  # lower-level split points

print("🎯 Target segmentation time points (actual boundaries):")
print(f"  Top level (2 events): {target_splits_top[0]//60:02d}:{target_splits_top[0]%60:02d}")
print(f"  Lower level ({len(target_splits_lower)+1} events): ", end="")
for t in target_splits_lower:
    print(f"{t//60:02d}:{t%60:02d}", end=" ")
print("\n")

# Parameter search range
ws_range = range(5, 35, 3)
thr_range = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
sigma_range = [10, 20, 30, 40, 50, 60, 80, 100, 120]
tolerance = 20  # Allowed time error (seconds)

print("🔧 Parameter search range:")
print(f"  Window size (ws): {list(ws_range)}")
print(f"  Threshold (thr): {thr_range}")
print(f"  SCM temporal kernel bandwidth (sigma): {sigma_range}")
print(f"  Tolerance: ±{tolerance} seconds")
print()



print("🔄 Starting parameter search (matching actual boundary frames)...")
print()

best_params_boundary = None
best_error_boundary = float('inf')
best_results_boundary = None
all_results_boundary = []
error_count = 0

total_combinations = len(ws_range) * len(thr_range) * len(sigma_range)
tested = 0

for ws, thr, sigma in itertools.product(ws_range, thr_range, sigma_range):
    tested += 1
    if tested % 50 == 0:
        print(f"  Tested {tested}/{total_combinations} parameter combinations... (Found {len(all_results_boundary)} valid combinations)")
    
    try:
        # 1. Atomic Segmentation
        scaler_loop = StandardScaler()
        feats_scaled_loop = scaler_loop.fit_transform(features)
        
        min_seg = max(FPS, ws)
        sw_boundaries, _, _ = sw_asymmetric(
            feats_scaled_loop, window_size=ws, step_reset=1,
            threshold=thr, similarity_method='cosine', min_segment_frames=min_seg
        )
        
        div_boundaries = traditional_diff_peaks(feats_scaled_loop, FPS)
        boundaries_test = post_fuse_and_refine(sw_boundaries, div_boundaries, feats_scaled_loop, FPS)
        
        # Convert to segment format
        segs_test = []
        prev = 0
        for b in boundaries_test:
            segs_test.append((prev, b))
            prev = b + 1
        segs_test.append((prev, features.shape[0] - 1))
        
        if len(segs_test) < 2:
            continue
        
        # 2. Build atomic features and centers
        atomic_features_test = []
        atomic_centers_test = []
        for start, end in segs_test:
            seg_feat = features[start:end+1].mean(axis=0)
            atomic_features_test.append(seg_feat)
            atomic_centers_test.append((start + end) / 2 / FPS)
        
        atomic_features_test = np.array(atomic_features_test)
        atomic_centers_test = np.array(atomic_centers_test)
        K_test = len(segs_test)
        
        # 3. Build SCM
        cos_sim_test = atomic_features_test @ atomic_features_test.T
        time_diff_test = np.abs(atomic_centers_test[:, None] - atomic_centers_test[None, :])
        temporal_kernel_test = np.exp(-time_diff_test**2 / (2 * sigma**2))
        SCM_test = cos_sim_test * temporal_kernel_test
        
        # 4. Hierarchical Clustering
        dist_matrix_test = 1 - SCM_test
        np.fill_diagonal(dist_matrix_test, 0)
        condensed_dist_test = dist_matrix_test[np.triu_indices(K_test, k=1)]
        Z_test = linkage(condensed_dist_test, method='ward')
        
        # Test top level (2 events) - Using actual boundary frames
        labels_top = fcluster(Z_test, t=2, criterion='maxclust')
        
        # Find top level split points (Using actual boundary frames instead of centers)
        top_splits = []
        for i in range(1, K_test):
            if labels_top[i] != labels_top[i-1]:
                boundary_frame = segs_test[i][0]  # Using start frame
                boundary_time = int(boundary_frame / FPS)
                top_splits.append(boundary_time)
        
        # Calculate top level error
        top_error = calculate_split_error(top_splits, target_splits_top, tolerance)
        
        # Test multiple lower levels
        best_lower_error = float('inf')
        best_lower_splits = []
        best_num_events = 0
        
        for num_events_lower in range(8, 16):
            try:
                labels_lower = fcluster(Z_test, t=num_events_lower, criterion='maxclust')
                
                # Find lower level split points (Using actual boundary frames)
                lower_splits = []
                for i in range(1, K_test):
                    if labels_lower[i] != labels_lower[i-1]:
                        boundary_frame = segs_test[i][0]  # Using start frame
                        boundary_time = int(boundary_frame / FPS)
                        lower_splits.append(boundary_time)
                
                # Calculate lower level error
                lower_error = calculate_split_error(lower_splits, target_splits_lower, tolerance)
                
                if lower_error < best_lower_error:
                    best_lower_error = lower_error
                    best_lower_splits = lower_splits
                    best_num_events = num_events_lower
            except:
                continue
        
        if best_lower_error == float('inf'):
            continue
        
        lower_error = best_lower_error
        lower_splits = best_lower_splits
        
        # Total error
        total_error = top_error * 3 + lower_error
        
        # Save all valid results
        all_results_boundary.append({
            'params': {'ws': ws, 'thr': thr, 'sigma': sigma},
            'top_splits': top_splits,
            'lower_splits': lower_splits,
            'top_error': top_error,
            'lower_error': lower_error,
            'total_error': total_error,
            'num_atomic': K_test,
            'num_events_lower': best_num_events
        })
        
        if total_error < best_error_boundary:
            best_error_boundary = total_error
            best_params_boundary = {'ws': ws, 'thr': thr, 'sigma': sigma}
            best_results_boundary = {
                'top_splits': top_splits,
                'lower_splits': lower_splits,
                'top_error': top_error,
                'lower_error': lower_error,
                'num_atomic': K_test
            }
    
    except Exception as e:
        error_count += 1
        continue

print()
print("="*70)
print("✅ Parameter search complete (matching actual boundary frames)!")
print("="*70)
print()

print(f"📊 Statistics:")
print(f"  Test combinations: {tested}")
print(f"  Exception count: {error_count}")
print(f"  Valid combinations: {len(all_results_boundary)}")
print()

if best_params_boundary is not None:
    print("🎯 Best parameters (matching actual boundary frames):")
    print(f"  Sliding window size (ws): {best_params_boundary['ws']}")
    print(f"  Threshold (thr): {best_params_boundary['thr']:.2f}")
    print(f"  SCM time kernel bandwidth (sigma): {best_params_boundary['sigma']} seconds")
    print(f"  Number of atomic units: {best_results_boundary['num_atomic']}")
    print()
    
    print("📊 Split Results Comparison (Actual Boundary Frames):")
    print()
    print("  Top Level (2 events):")
    print(f"    Target split points: {target_splits_top[0]//60:02d}:{target_splits_top[0]%60:02d}")
    if best_results_boundary['top_splits']:
        for t in best_results_boundary['top_splits']:
            print(f"    Actual split points: {t//60:02d}:{t%60:02d}")
        print(f"    Error: {best_results_boundary['top_error']:.1f} seconds")
    print()
    
    print(f"  Lower Level ({len(best_results_boundary['lower_splits'])+1} events):")
    print("    Target split points: ", end="")
    for t in target_splits_lower:
        print(f"{t//60:02d}:{t%60:02d}", end=" ")
    print()
    print("    Actual split points: ", end="")
    for t in best_results_boundary['lower_splits']:
        print(f"{t//60:02d}:{t%60:02d}", end=" ")
    print()
    print(f"    Average Error: {best_results_boundary['lower_error']:.1f} seconds")
    print()
    
    print("💡 Comparison:")
    print("  Original (matching atomic centers): ws=14, thr=0.50, sigma=80 -> center 02:27, boundary 02:11")
    print(f"  Improved (matching actual boundaries): ws={best_params_boundary['ws']}, thr={best_params_boundary['thr']:.2f}, sigma={best_params_boundary['sigma']} -> boundary {best_results_boundary['top_splits'][0]//60:02d}:{best_results_boundary['top_splits'][0]%60:02d}")
else:
    print("❌ No exact match for the parameter combination was found")
    
    if all_results_boundary:
        print()
        print("📋 Most similar parameter combinations (sorted by error):")
        print()
        all_results_sorted = sorted(all_results_boundary, key=lambda x: x['total_error'])
        
        for i, result in enumerate(all_results_sorted[:10], 1):
            params = result['params']
            print(f"  {i}. ws={params['ws']}, thr={params['thr']:.2f}, sigma={params['sigma']}")
            print(f"     Atomic units: {result['num_atomic']}")
            print(f"     Top-level error: {result['top_error']:.1f} seconds, Lower-level error: {result['lower_error']:.1f} seconds")
            print(f"     Top-level splits: ", end="")
            for t in result['top_splits']:
                print(f"{t//60:02d}:{t%60:02d}", end=" ")
            print(f" (Target: {target_splits_top[0]//60:02d}:{target_splits_top[0]%60:02d})")
            print()

print()